# Preprocess the control data from GSE37069

We do this so that it is in the right format to use in the process of network diffusion

In [2]:
from pathlib import Path
from io import StringIO
import re
import numpy as np
import pandas as pd

# =====================================================
# PATHS
# =====================================================
PROJECT_ROOT = Path("/Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries")
BURN_DATA_DIR = PROJECT_ROOT / "preprocessing" / "data" / "GSE37069"

SERIES_PATH = BURN_DATA_DIR / "Burn_GSE37069_series_matrix.txt"
ANNOT_PATH  = BURN_DATA_DIR / "Annotation_GPL570-55999.txt"

OUT_DIR = PROJECT_ROOT / "preprocessing" / "burn_control" / "preprocessed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================
# READERS
# =====================================================
def read_affy_gpl570_annotation(path: Path) -> pd.DataFrame:
    path = Path(path)
    df = pd.read_csv(path, sep="\t", comment="#", dtype=str, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    for c in df.columns:
        df[c] = df[c].astype(str).str.strip().str.strip('"')
        df.loc[df[c].isin(["nan", "NaN", "None"]), c] = ""

    required = ["ID", "Gene Symbol"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Annotation missing columns {missing}. Found columns: {df.columns.tolist()[:30]}")

    if "SPOT_ID" not in df.columns:
        df["SPOT_ID"] = ""

    # Option A: keep first symbol before ///
    df["Gene Symbol"] = (
        df["Gene Symbol"]
        .fillna("")
        .astype(str)
        .str.split("///").str[0]
        .str.strip()
    )

    return df


def read_geo_series_matrix_full(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    path = Path(path)
    meta_rows = []
    table_lines = []
    in_table = False

    with path.open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("!series_matrix_table_begin"):
                in_table = True
                continue
            if line.startswith("!series_matrix_table_end"):
                in_table = False
                continue

            if in_table:
                table_lines.append(line)
            elif line.startswith("!"):
                meta_rows.append(line)

    meta_parsed = []
    for row in meta_rows:
        parts = row.split("\t")
        key = parts[0].lstrip("!")
        values = parts[1:]
        meta_parsed.append((key, values))

    metadata_df = pd.DataFrame(
        {"key": [k for k, _ in meta_parsed], "values": [v for _, v in meta_parsed]}
    )

    if not table_lines:
        raise ValueError("No series matrix table found between begin/end markers.")

    expr_df = pd.read_csv(StringIO("\n".join(table_lines)), sep="\t", dtype=str, low_memory=False)
    expr_df.columns = [c.strip().strip('"') for c in expr_df.columns]
    for c in expr_df.columns:
        expr_df[c] = expr_df[c].astype(str).str.strip().str.strip('"')
        expr_df.loc[expr_df[c].isin(["nan", "NaN", "None"]), c] = ""

    return metadata_df, expr_df

# =====================================================
# METADATA PARSING
# =====================================================
def parse_kv_cell(s):
    if s is None:
        return None
    s = str(s).strip().strip('"')
    if ":" not in s:
        return None
    k, v = s.split(":", 1)
    return k.strip().lower(), v.strip()


def parse_hours_since_injury(x):
    if x is None:
        return None
    s = str(x).strip().strip('"')
    if s == "" or s.lower() in {"na", "n/a", "nan", "none"}:
        return None
    m = re.search(r"(\d+(\.\d+)?)", s)
    if not m:
        return None
    return float(m.group(1))


def assign_time_bin(time_hours):
    if time_hours is None:
        return None
    if time_hours == 0:
        return "T0"
    if 1 <= time_hours < 24:
        return "Early"
    if 24 <= time_hours < 96:
        return "Mid"
    if 96 <= time_hours < 168:
        return "Late"
    if time_hours >= 169:
        return "FollowUp"
    return "Other"


def assign_phase_bucket(time_bin):
    if time_bin is None:
        return None
    if time_bin in {"T0", "Early", "Mid"}:
        return "Acute"
    if time_bin == "Late":
        return "Proliferation"
    if time_bin == "FollowUp":
        return "Remodelling"
    return None


def extract_sample_metadata(meta_df: pd.DataFrame) -> pd.DataFrame:
    geo_acc_rows = meta_df[meta_df["key"].str.lower() == "sample_geo_accession"]
    if geo_acc_rows.empty:
        raise ValueError("Could not find !Sample_geo_accession in metadata.")
    samples = [str(x).strip().strip('"') for x in geo_acc_rows.iloc[0]["values"]]
    per_sample = {gsm: {} for gsm in samples}

    src_rows = meta_df[meta_df["key"].str.lower() == "sample_source_name_ch1"]
    if not src_rows.empty:
        values = src_rows.iloc[0]["values"]
        n = min(len(values), len(samples))
        for i in range(n):
            per_sample[samples[i]]["source_name"] = str(values[i]).strip().strip('"')

    char_rows = meta_df[
        meta_df["key"].str.lower().isin(["sample_characteristics_ch1", "sample_characteristics_ch2"])
    ]
    for _, row in char_rows.iterrows():
        values = row["values"]
        n = min(len(values), len(samples))
        for i in range(n):
            kv = parse_kv_cell(values[i])
            if kv is None:
                continue
            k, v = kv
            per_sample[samples[i]][k] = v

    meta = pd.DataFrame.from_dict(per_sample, orient="index")
    meta.index.name = "sample_id"

    if "hours_since_injury" in meta.columns:
        meta["time_hours"] = meta["hours_since_injury"].apply(parse_hours_since_injury)
    else:
        meta["time_hours"] = np.nan

    meta["time_bin"] = meta["time_hours"].apply(assign_time_bin)
    meta["phase_bucket"] = meta["time_bin"].apply(assign_phase_bucket)

    if "age" in meta.columns:
        meta["age"] = pd.to_numeric(meta["age"], errors="coerce")

    return meta

# =====================================================
# EXPRESSION PREPROCESSING
# =====================================================
def preprocess_affy_gpl570(expr_df: pd.DataFrame, anno_df: pd.DataFrame) -> pd.DataFrame:
    expr = expr_df.copy().rename(columns={"ID_REF": "probe_id"})
    expr["probe_id"] = expr["probe_id"].astype(str).str.strip()

    anno = anno_df.copy().rename(columns={"ID": "probe_id", "Gene Symbol": "gene_symbol"})
    anno["probe_id"] = anno["probe_id"].astype(str).str.strip()
    anno["gene_symbol"] = anno["gene_symbol"].astype(str).str.strip().str.upper()

    merged = expr.merge(anno[["probe_id", "gene_symbol", "SPOT_ID"]], on="probe_id", how="left")

    merged = merged[merged["SPOT_ID"].fillna("").astype(str).str.strip() == ""]
    merged = merged[merged["gene_symbol"].fillna("").astype(str).str.len() > 0]

    sample_cols = [c for c in merged.columns if c not in {"probe_id", "gene_symbol", "SPOT_ID"}]
    for c in sample_cols:
        merged[c] = pd.to_numeric(merged[c], errors="coerce")

    merged["_var"] = merged[sample_cols].var(axis=1, ddof=1)
    idx = merged.groupby("gene_symbol")["_var"].idxmax()
    gene_mat = merged.loc[idx].set_index("gene_symbol")[sample_cols]

    # normalize index consistently
    gene_mat.index = gene_mat.index.astype(str).str.strip().str.upper()

    return gene_mat

# =====================================================
# RUN
# =====================================================
print("[INFO] Loading annotation:", ANNOT_PATH)
anno_df = read_affy_gpl570_annotation(ANNOT_PATH)

print("[INFO] Loading series matrix:", SERIES_PATH)
meta_df, expr_df = read_geo_series_matrix_full(SERIES_PATH)

meta = extract_sample_metadata(meta_df)

if "source_name" not in meta.columns:
    raise KeyError("source_name column not found in metadata after extraction.")

control_mask = meta["source_name"].astype(str).str.strip().str.lower() == "control"
control_samples = meta.index[control_mask].astype(str).tolist()

print(f"[INFO] Total samples in metadata: {meta.shape[0]}")
print(f"[INFO] Control samples detected from sample_source_name_ch1: {len(control_samples)}")

gene_mat = preprocess_affy_gpl570(expr_df, anno_df)

control_samples = [s for s in control_samples if s in gene_mat.columns]
gene_mat_ctrl = gene_mat[control_samples]

meta_ctrl = meta.loc[control_samples].copy()

# Controls have no injury time -> assign single timepoint "Ctrl"
meta_ctrl["time_bin"] = "Ctrl"

meta_ctrl_time = meta_ctrl.copy()
gene_mat_ctrl_time = gene_mat_ctrl

print(f"[INFO] Controls found in expression: {len(control_samples)}")
print(f"[INFO] Controls with valid time_hours: {meta_ctrl_time.shape[0]}")
print(f"[INFO] Control gene matrix shape (all controls): {gene_mat_ctrl.shape}")
print(f"[INFO] Control gene matrix shape (timepoint-valid controls): {gene_mat_ctrl_time.shape}")

genes_x_samples_out = OUT_DIR / "GSE37069_controls__genes_x_samples.tsv"
gene_mat_ctrl.to_csv(genes_x_samples_out, sep="\t")

meta_out = OUT_DIR / "GSE37069_controls__sample_metadata.tsv"
meta_ctrl.to_csv(meta_out, sep="\t")

tp_labels = meta_ctrl_time["time_bin"].astype(str)
tp_counts = tp_labels.value_counts()

pb = {}
for tp in tp_counts.index:
    cols = tp_labels[tp_labels == tp].index
    pb[tp] = gene_mat_ctrl_time[cols].mean(axis=1, skipna=True)

pb_df = pd.DataFrame(pb)

pseudobulk_out = OUT_DIR / "GSE37069_controls__pseudobulk_genes_x_timepoint.tsv"
pb_df.to_csv(pseudobulk_out, sep="\t")

tp_counts_out = OUT_DIR / "GSE37069_controls__timepoint_counts.tsv"
tp_counts.to_csv(tp_counts_out, sep="\t")

print("\n===== SUMMARY (GSE37069 controls) =====")
print("Controls detected (metadata):", int(control_mask.sum()))
print("Controls found in expression:", len(control_samples))
print("Controls kept for pseudobulk (valid time_hours):", meta_ctrl_time.shape[0])
print("Genes:", gene_mat_ctrl.shape[0])
print("Time bins present:", tp_counts.to_dict())
print("[SAVED]", genes_x_samples_out)
print("[SAVED]", pseudobulk_out)
print("[SAVED]", tp_counts_out)
print("[SAVED]", meta_out)
print("======================================\n")

[INFO] Loading annotation: /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/preprocessing/data/GSE37069/Annotation_GPL570-55999.txt
[INFO] Loading series matrix: /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/preprocessing/data/GSE37069/Burn_GSE37069_series_matrix.txt
[INFO] Total samples in metadata: 590
[INFO] Control samples detected from sample_source_name_ch1: 37


/var/folders/sq/c9xq2zk97n3cf_nqf5x49c0w0000gn/T/ipykernel_41173/638221196.py:209: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged["_var"] = merged[sample_cols].var(axis=1, ddof=1)


[INFO] Controls found in expression: 37
[INFO] Controls with valid time_hours: 37
[INFO] Control gene matrix shape (all controls): (19923, 37)
[INFO] Control gene matrix shape (timepoint-valid controls): (19923, 37)

===== SUMMARY (GSE37069 controls) =====
Controls detected (metadata): 37
Controls found in expression: 37
Controls kept for pseudobulk (valid time_hours): 37
Genes: 19923
Time bins present: {'Ctrl': 37}
[SAVED] /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/preprocessing/burn_control/preprocessed/GSE37069_controls__genes_x_samples.tsv
[SAVED] /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/preprocessing/burn_control/preprocessed/GSE37069_controls__pseudobulk_genes_x_timepoint.tsv
[SAVED] /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/preprocessing/burn_control/preprocessed/GSE37069_controls__timepoint_counts.tsv
[SAVED] /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/preprocessing/burn_control/pr